In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
import pickle

In [2]:
dataset = pd.read_csv("naukri_data_science_jobs_india.csv")
dataset.head(9)

,Job_Role,Company,Location,Job Experience,Skills/Description
0,Senior Data Scientist,UPL,"Bangalore/Bengaluru, Mumbai (All Areas)",3-6,"python, MLT, statistical modeling, machine lea..."
1,Senior Data Scientist,Walmart,Bangalore/Bengaluru,5-9,"Data Science, Machine learning, Python, Azure,..."
2,Applied Data Scientist / ML Senior Engineer (P...,SAP India Pvt.Ltd,Bangalore/Bengaluru,5-10,"Python, IT Skills, Testing, Cloud, Product Man..."
3,Data Scientist,UPL,"Bangalore/Bengaluru, Mumbai (All Areas)",1-4,"python, machine learning, Data Science, data a..."
4,Data Scientist,Walmart,Bangalore/Bengaluru,4-8,"IT Skills, Python, Data Science, Machine Learn..."
5,Principal Data Scientist,Walmart,Bangalore/Bengaluru,7-12,"Data Science, oral communication, Shell, Pytho..."
6,Data Scientist,Walmart,Bangalore/Bengaluru,4-8,"Computer science, cassandra, Machine learning,..."
7,Expert Data Scientist,UPL,"Bangalore/Bengaluru, Mumbai (All Areas)",6-9,"team lead, MLT, machine learning, aws, Python,..."
8,Data Scientist,Ericsson,Bangalore/Bengaluru,10-20,"Graphics, Bidding, Google Analytics, Data mana..."


In [3]:
print(dataset.shape)
print(dataset.columns)
print(dataset.info())

(12000, 5)
Index(['Job_Role', 'Company', 'Location', 'Job Experience',
       'Skills/Description'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Job_Role            12000 non-null  str  
 1   Company             12000 non-null  str  
 2   Location            12000 non-null  str  
 3   Job Experience      12000 non-null  str  
 4   Skills/Description  12000 non-null  str  
dtypes: str(5)
memory usage: 2.3 MB
None


In [4]:
print(dataset.describe())

             Job_Role    Company             Location Job Experience  \
count           12000      12000                12000          12000   
unique           6563       3507                  822            143   
top     Data Engineer  Accenture  Bangalore/Bengaluru           5-10   
freq              580        490                 3383            944   

       Skills/Description  
count               12000  
unique              11356  
top        Data Scientist  
freq                    8  


In [5]:
dataset.isnull().sum()

Job_Role              0
Company               0
Location              0
Job Experience        0
Skills/Description    0
dtype: int64

In [6]:
dataset.drop_duplicates().sum()

Job_Role              Senior Data ScientistSenior Data ScientistAppl...
Company               UPLWalmartSAP India Pvt.LtdUPLWalmartWalmartWa...
Location              Bangalore/Bengaluru, Mumbai (All Areas)Bangalo...
Job Experience        3-65-95-101-44-87-124-86-910-205-97-122-72-73-...
Skills/Description    python, MLT, statistical modeling, machine lea...
dtype: str

In [7]:
dataset = dataset.dropna(subset=["Job Experience"])

In [8]:
dataset["Location"] = dataset["Location"].str.lower().str.strip()
dataset["Job_Role"] = dataset["Job_Role"].str.lower().str.strip()
dataset["Company"] = dataset["Company"].str.lower().str.strip()

dataset.head()

,Job_Role,Company,Location,Job Experience,Skills/Description
0,senior data scientist,upl,"bangalore/bengaluru, mumbai (all areas)",3-6,"python, MLT, statistical modeling, machine lea..."
1,senior data scientist,walmart,bangalore/bengaluru,5-9,"Data Science, Machine learning, Python, Azure,..."
2,applied data scientist / ml senior engineer (p...,sap india pvt.ltd,bangalore/bengaluru,5-10,"Python, IT Skills, Testing, Cloud, Product Man..."
3,data scientist,upl,"bangalore/bengaluru, mumbai (all areas)",1-4,"python, machine learning, Data Science, data a..."
4,data scientist,walmart,bangalore/bengaluru,4-8,"IT Skills, Python, Data Science, Machine Learn..."


In [9]:
dataset.dtypes

Job_Role              str
Company               str
Location              str
Job Experience        str
Skills/Description    str
dtype: object

In [27]:
def safe_parse(exp_str):
    try:
        parts = [int(p.strip()) for p in str(exp_str).split('-')]
        if len(parts) == 2:
            return parts[0], parts[1], sum(parts)/2
        return None, None, None
    except:
        return None, None, None

parsed = dataset['Job Experience'].apply(safe_parse)
dataset['Min_Exp'] = parsed.apply(lambda x: x[0])
dataset['Max_Exp'] = parsed.apply(lambda x: x[1])
dataset['Avg_Exp'] = parsed.apply(lambda x: x[2])
dataset = dataset.dropna(subset=['Avg_Exp'])

In [29]:
dataset.head(6)

,Job_Role,Company,Location,Job Experience,Skills/Description,Avg_Experience,Loc_ahmedabad,Loc_bangalore/bengaluru,"Loc_bangalore/bengaluru, delhi / ncr","Loc_bangalore/bengaluru, delhi / ncr, mumbai (all areas)",...,"Loc_pune, gurgaon/gurugram, bangalore/bengaluru","Loc_pune, mumbai (all areas)","Loc_pune, united states (usa)",Loc_remote,Loc_surat,Loc_trivandrum/thiruvananthapuram,Loc_vadodara,Min_Exp,Max_Exp,Avg_Exp
0,senior data scientist,upl,"bangalore/bengaluru, mumbai (all areas)",3-6,"python, MLT, statistical modeling, machine lea...",4.5,False,False,False,False,...,False,False,False,False,False,False,False,3,6,4.5
1,senior data scientist,walmart,bangalore/bengaluru,5-9,"Data Science, Machine learning, Python, Azure,...",7.0,False,True,False,False,...,False,False,False,False,False,False,False,5,9,7.0
2,applied data scientist / ml senior engineer (p...,sap india pvt.ltd,bangalore/bengaluru,5-10,"Python, IT Skills, Testing, Cloud, Product Man...",7.5,False,True,False,False,...,False,False,False,False,False,False,False,5,10,7.5
3,data scientist,upl,"bangalore/bengaluru, mumbai (all areas)",1-4,"python, machine learning, Data Science, data a...",2.5,False,False,False,False,...,False,False,False,False,False,False,False,1,4,2.5
4,data scientist,walmart,bangalore/bengaluru,4-8,"IT Skills, Python, Data Science, Machine Learn...",6.0,False,True,False,False,...,False,False,False,False,False,False,False,4,8,6.0
5,principal data scientist,walmart,bangalore/bengaluru,7-12,"Data Science, oral communication, Shell, Pytho...",9.5,False,True,False,False,...,False,False,False,False,False,False,False,7,12,9.5


In [32]:
skills = ['python','sql','machine learning','deep learning',
          'nlp','tableau','power bi','aws','azure','spark',
          'tensorflow','statistics','excel','docker']
desc = dataset['Skills/Description'].str.lower().fillna('')
for skill in skills:
    col = f'skill_{skill.replace(" ","_").replace("/","_")}'
    dataset[col] = desc.str.contains(skill).astype(int)
dataset['skill_count'] = dataset[[f'skill_{s.replace(" ","_").replace("/","_")}' for s in skills]].sum(axis=1)


In [36]:
le_loc  = LabelEncoder()
le_role = LabelEncoder()
dataset['loc_enc']  = le_loc.fit_transform(dataset['Location'])
dataset['role_enc'] = le_role.fit_transform(dataset['Job_Role'])


In [38]:
skill_cols = [f'skill_{s.replace(" ","_").replace("/","_")}' for s in skills]
features   = ['loc_enc','role_enc','skill_count','Min_Exp','Max_Exp'] + skill_cols

In [39]:
X = dataset[features]
y = dataset['Avg_Exp']

In [40]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [42]:
model = RandomForestRegressor(n_estimators=300,max_depth=15,random_state=42,n_jobs=-1)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [43]:
print(f"Train R²: {model.score(X_train,y_train)*100:.2f}%")
print(f"Test  R²: {model.score(X_test,y_test)*100:.2f}%")

Train R²: 100.00%
Test  R²: 99.95%


In [46]:
with open('model.pkl','wb') as f: pickle.dump(model, f)
with open('encoders.pkl','wb') as f:
    pickle.dump({'loc':le_loc,'role':le_role,'skills':skills}, f)
print("Saved model.pkl and encoders.pkl")

Saved model.pkl and encoders.pkl
